In [ ]:
%%html
<style>
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from time import time

from pathlib import Path
from dahuffman import HuffmanCodec

basic_value = 0

folder_type = "raif"  # "trans_34k"  "transact_10k"

processed_folder = Path(f"processed_data/{folder_type}")
folder = Path(f"data/{folder_type}")
files = [f for f in folder.iterdir() if f.is_file()]
files.sort()
files

In [ ]:
column_mapping_based_on_folder_name = {
    "transact_18_22": {
        "client": "client",
        "mcc": "mcc",
        "date": "date",
        "amount": "amt",
    },
    "trans_600k": {
        "client": "CustomerKey",
        "mcc": "MCC",
        "date": "DATE",
        "amount": "AMOUNT_EQ",
    },
    "raif": {
        "client": "client",
        "mcc": "mcc",
        "date": "date",
        "amount": "amt",
    },
}

In [ ]:
dates = []
unique_clients = set()

In [ ]:
mcc_to_category = pd.read_csv("data/mcc_to_category.csv", index_col="mcc")


def fill_missing_dates(df):
    df.index = df.index.set_levels(pd.to_datetime(df.index.levels[1]), level=1)
    min_date, max_date = (
        df.index.get_level_values("date").min(),
        df.index.get_level_values("date").max(),
    )
    dates.append((min_date, max_date))

    full_range = pd.date_range(min_date, max_date, freq="D")
    clients_grouped = df.groupby(level=0)

    for client, group in tqdm(
        clients_grouped,
        total=len(clients_grouped),
        desc="Filling missing dates",
        position=2,
        leave=False,
    ):
        idx = pd.to_datetime(group.index.get_level_values("date"))
        missing_dates = full_range.difference(idx)

        if len(missing_dates) == 0:
            continue

        new_index = pd.MultiIndex.from_product(
            [[client], missing_dates], names=df.index.names
        )

        new_rows = pd.DataFrame(0, index=new_index, columns=df.columns)
        df = pd.concat([df, new_rows])
    return df


def preprocess(df):
    df["client"] = df["client"].astype(int)
    df = df.dropna(subset=["mcc"])
    df["mcc"] = df["mcc"].astype(int)

    df["category"] = df["mcc"].map(mcc_to_category["category"])
    df = df.dropna(subset=["category"])
    df = df.drop(columns=["mcc"])
    df = df.groupby(["client", "date"]).agg(list)

    surv = np.zeros(len(df), dtype=int)
    soc = np.zeros(len(df), dtype=int)
    selfr = np.zeros(len(df), dtype=int)
    code = np.zeros(len(df), dtype=int)

    for i, row in enumerate(
        tqdm(
            df.itertuples(index=False),
            total=len(df),
            desc="Creating SSSR",
            position=1,
            leave=False,
        )
    ):
        tmp = {"survival": 0, "socialization": 0, "self_realization": 0}

        for value, category in zip(row.amount, row.category):
            tmp[category] += value

        s = int(tmp["survival"] > basic_value)
        so = int(tmp["socialization"] > basic_value)
        sr = int(tmp["self_realization"] > basic_value)

        surv[i] = s
        soc[i] = so
        selfr[i] = sr
        code[i] = s * 4 + so * 2 + sr

    df["survival"] = surv
    df["socialization"] = soc
    df["self_realization"] = selfr
    df["code"] = code

    df = df[["survival", "socialization", "self_realization", "code"]]

    df = fill_missing_dates(df)
    df = df.sort_index()

    return df

In [ ]:
def LempelZiv(S):
    n = len(S)
    i = 0
    C = u = v = vmax = 1
    while (u + v) < n:
        if S[i + v] == S[u + v]:
            v += 1
        else:
            vmax = max(v, vmax)
            i += 1
            v = 1
            if i == u:
                C += 1
                u += vmax
                i = 0
                vmax = v
            else:
                v = 1
    if v != 1:
        C += 1
    return C


def LZW_compress(text, dict_size=8):
    dictionary = {str(i): i for i in range(dict_size)}
    compessed_data = []
    string = ""
    for symbol in text:
        new_string = string + symbol
        if new_string in dictionary:
            string = new_string
        else:
            compessed_data.append(dictionary[string])
            dictionary[new_string] = dict_size
            dict_size += 1
            string = symbol
    if string in dictionary:
        compessed_data.append(dictionary[string])
    return compessed_data, dictionary

In [ ]:
def compute_lzc(df: pd.DataFrame, columns: list):
    clients = df.index.get_level_values(0).unique()
    result = []

    for client in tqdm(clients, desc="Computing LZC", position=2, leave=False):
        client_data = df.xs(client, level=0)
        client_row = {}

        for col in columns:
            s = "".join(client_data[col].astype(str).tolist())
            lzc = LempelZiv(s)

            client_row[f"{col}"] = lzc

        code_text = "".join(client_data["code"].astype(str).tolist())
        compressed_text, dictionary = LZW_compress(code_text, 8)
        client_row[f"LZW"] = (
            len(compressed_text) / len(code_text) / 3
        )  # + len(dictionary)
        result.append(pd.Series(client_row, name=client))

    df_result = pd.DataFrame(result)
    df_result.index.name = "client"

    return df_result

In [ ]:
def compute_huffman(df: pd.DataFrame, columns: list):
    clients = df.index.get_level_values(0).unique()
    results = []

    for client in tqdm(
        clients, desc="Computing Huffman codes", position=3, leave=False
    ):
        client_data = df.xs(client, level=0)
        row = {}

        for col in columns:
            multiplier = 3 if col == "code" else 1
            seq = client_data[col].astype(int).tolist()
            codec = HuffmanCodec.from_data(seq)
            code_table = codec.get_code_table()
            compressed_bits = 0
            for symbol in seq:
                bitsize, _ = code_table[symbol]
                compressed_bits += bitsize

            original_bits = len(seq) * multiplier
            ratio = compressed_bits / original_bits
            row[f"huffman_ratio_{col}"] = ratio

        results.append(pd.Series(row, name=client))

    df_result = pd.DataFrame(results)
    df_result.index.name = "client"
    return df_result

In [ ]:
time_intervals = {  # start_date,   end_date
    "trans_600k": ("2017-05-01", "2019-01-31"),
    "transact_18_22": ("2018-01-01", "2019-12-14"),
    "raif": ("2017-04-01", "2017-09-30"),
}

In [ ]:
processed_folder.mkdir(parents=True, exist_ok=True)

compression_times = []

for file in tqdm(files, desc="Processing files", position=0):
    mapping = column_mapping_based_on_folder_name[folder.name]
    df = pd.read_csv(file, usecols=list(mapping.values()))
    df.rename(columns={v: k for k, v in mapping.items()}, inplace=True)
    print(df["date"].min(), df["date"].max())

    start_date, end_date = time_intervals[folder_type]
    mask = (df["date"] >= start_date) & (df["date"] <= end_date)
    df = df.loc[mask]

    processed_df = preprocess(df)
    processed_df.to_csv(processed_folder.joinpath(file.name), index=True)

    start = time()
    lzc = compute_lzc(processed_df, ["survival", "socialization", "self_realization"])
    lzc.to_csv(
        processed_folder.joinpath(file.name.replace(".csv", "_lzc.csv")), index=True
    )
    huffman_df = compute_huffman(processed_df, ["code"])
    huffman_df.to_csv(
        processed_folder.joinpath(file.name.replace(".csv", "_huffman.csv")), index=True
    )
    end = time()
    compression_times.append(
        (end - start) / len(processed_df.index.get_level_values(0).unique())
    )

    # sanity checks
    # check unique clients
    clients = df["client"].unique().tolist()
    start_len = len(unique_clients)
    unique_clients.update(clients)

    if start_len + len(clients) != len(unique_clients):
        raise ValueError(f"Duplicate user IDs found: {file}")

    # check same dates
    if len(dates) >= 2:
        if dates[-1][0] != dates[-2][0] or dates[-1][1] != dates[-2][1]:
            raise ValueError(f"Date ranges do not match: {file} {dates}")
    print(dates[0])

In [ ]:
sum(compression_times) / len(compression_times)

In [ ]:
len(unique_clients)